# 📘 Explained copy — 00 Setup, assumptions and data audit

This is a **study copy** of `notebooks/00_setup_and_data_audit.ipynb`. Every original cell is kept unchanged
(with its saved output). Above each cell there is a blue-quoted **Explanation** block with:

- **What it does** – plain-English summary of the cell
- **Why this method** – why this approach was chosen over the alternatives
- **Input → Output** – what goes in, what comes out
- **How to read the output / why it matters** – what the number means for the report and for later modules

**Role of this notebook in the project.** It is the foundation: it fixes the shared assumptions, checks all eight
data files for quality, decides the money units, decides the validation scheme for each dataset (random vs time
split) and lists leakage risks. Every later design choice (PR-AUC for fraud, walk-forward for equities, day-split
for HFT, etc.) is justified by a finding in this notebook.

> Note: the outputs were saved on another machine, so paths printed (e.g. `/mnt/user-data/...`) differ from yours.

> ### 📘 Explanation — cell 0
> **What it does.** Title cell + the "pipeline map": each module notebook saves a small result file, and notebook 10 reads them all.
>
> **Why this design.** A *hub-and-spoke* design (independent modules → one allocator) means each module can be run,
> checked and explained on its own, and the final ₹1,000 crore allocation is reproducible from the saved files rather
> than from copy-pasted numbers.
>
> **Report use.** If asked "how do the modules connect?" — this table is the answer: modules produce *risk-adjusted
> return estimates*; notebook 10 turns them into an allocation.

# 00. Setup, assumptions and data audit

**Purpose.** Everything every other notebook relies on: the shared assumptions, the units, and a quality check of all eight data files. Run this first.

**How the notebooks connect.** Each module notebook ends by saving one small file in `results/`. Notebook 10 reads them all and builds the ₹1,000 crore allocation.

| Step | Notebook | Saves |
|---|---|---|
| 1 | 01 Strategic initiatives | `results/initiatives.json` |
| 2 | 02 Credit risk | `results/credit.json` |
| 3 | 03 Fraud | `results/fraud.json` |
| 4 | 04 Churn and CLV | `results/churn.json` |
| 5 | 05 Equity returns | `results/equity_predictions.csv` |
| 6 | 06 Portfolio | `results/portfolio.json` |
| 7 | 07 HFT | `results/hft.json` |
| 8 | 08 Options | `results/options.json` |
| 9 | 09 Liquidity | `results/liquidity.json` |
| 10 | 10 Allocation | final tables |

> ### 📘 Explanation — cell 1
> **What it does.** Adds `../src` to Python's search path and imports everything from `novafin.py`
> (pandas `pd`, numpy `np`, matplotlib, seaborn, metrics, `load()`, `save_result()`, `ASSUMPTIONS`, `SEED`, …).
> The two `set_option` calls just make wide tables print fully.
>
> **Why this method.** One shared helper module instead of repeating imports/constants in 11 notebooks:
> same seed (42), same assumptions, same metric functions → results are consistent and comparable across modules.
>
> **Input → Output.** Input: nothing. Output: the message `NovaFin helpers loaded. Root: ...` confirming the project
> root (the folder with `data/` and `src/`) was found.

In [1]:
import sys; sys.path.insert(0, '../src')
from novafin import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

NovaFin helpers loaded. Root: /mnt/user-data/outputs/novafin_capstone


> ### 📘 Explanation — cell 2
> **Section header.** Shared assumptions live in one dictionary in `src/novafin.py`, so a change propagates to every module.

## 1. Shared assumptions
These are used by every module. Change them here (in `src/novafin.py`) and re-run the notebooks to see the effect everywhere.

> ### 📘 Explanation — cell 3
> **What it does.** Shows the `ASSUMPTIONS` dictionary as a one-column table.
>
> **Input → Output.** Input: `ASSUMPTIONS` dict. Output: 10 rows — LGD 0.40, discount rate 12%, risk-free 7%, VaR 99%,
> cost of funds 6%, max ECL ratio 3%, fraud miss cost ₹10,000, false-positive cost ₹500, capital ₹1,000 cr,
> retention budget 1,000 customers.
>
> **Where each is used later.**
> | Assumption | Used in |
> |---|---|
> | LGD 0.40 | ECL = PD × LGD × EAD (02 credit, 10 allocation) |
> | DISCOUNT_RATE 12% | NPV / IRR hurdle (01 initiatives) |
> | RISK_FREE 7% | Sharpe/Sortino (05, 06), options comparison (08) |
> | VAR_CONFIDENCE 99% | VaR in portfolio / liquidity stress (06, 09) |
> | COST_OF_FUNDS 6% | lending margin (02) |
> | MAX_ECL_RATIO 3% | approval cut-off for loan book (02) |
> | Fraud costs | cost-based threshold (03) |
> | RETENTION_BUDGET | top-1,000 customers to contact (04) |
>
> **Why it matters.** Examiners often ask "which numbers are given vs assumed?" — LGD, fraud costs, ₹1,000 cr and
> 1,000 contacts are **given in the brief**; 12%, 7%, 6%, 3% and 99% are **our assumptions** (next cell explains why).

In [2]:
pd.DataFrame({'Value': ASSUMPTIONS}).rename_axis('Assumption')

,Value
Assumption,
LGD,0.40
DISCOUNT_RATE,0.12
RISK_FREE,0.07
VAR_CONFIDENCE,0.99
COST_OF_FUNDS,0.06
MAX_ECL_RATIO,0.03
FRAUD_MISS_COST,10000.00
FRAUD_FALSE_POSITIVE_COST,500.00
TOTAL_CAPITAL_CR,1000.00


> ### 📘 Explanation — cell 4
> **Why these values (talking points).** 12% ≈ Indian corporate hurdle rate; 7% ≈ 10-year G-sec yield;
> 99% VaR = Basel-style confidence; 3% ECL cap = a risk-appetite statement. Being explicit that these are assumptions
> (and could be sensitivity-tested) is a strength in the report.

**Why these values.**
- **LGD 40%, fraud costs ₹10,000 / ₹500, ₹1,000 crore, 1,000 contacts:** given in the brief.
- **Discount rate 12%:** a typical hurdle rate for corporate projects in India. Used for NPV in notebook 01.
- **Risk-free 7%:** roughly the Indian 10-year government bond yield. Used for Sharpe ratios.
- **Cost of funds 6%:** funding cost for lending margins (notebook 02).
- **Max ECL ratio 3%:** the risk appetite for the loan book: expected credit loss should not exceed 3% of approved exposure.
- **VaR 99%:** a standard regulatory-style confidence level.

These are assumptions, not facts from the data, and the report should say so.

> ### 📘 Explanation — cell 5
> **Section header** – data quality audit of all 8 files.

## 2. Data inventory and quality checks

> ### 📘 Explanation — cell 6
> **What it does.** Loads all eight datasets with `load(name)` and, for each, records rows, columns, missing cells,
> duplicate rows and number of numeric columns. Keeps the DataFrames in a dict `data` for the later cells.
>
> **Why this method.** A single inventory table is the standard first step of any ML project (data audit): it tells you
> whether you need imputation, de-duplication or type fixing *before* modelling. Using `isna().sum().sum()` and
> `duplicated().sum()` is the simplest complete check.
>
> **Input → Output.** Input: 8 CSV files in `data/`. Output: inventory table — e.g. loans 5,000 rows, transactions
> 30,000, market 21,855, HFT 120,000 rows × 33 columns; **0 missing and 0 duplicates everywhere**.
>
> **How to read it.** Because nothing is missing, no imputation choices (mean/median/model) are needed → cleaning
> focuses on units, types and leakage instead. The row counts also show sample sizes: e.g. with 30,000 transactions
> and 2.3% fraud there are only ~690 fraud cases → this drives the need for stratified CV.

In [3]:
names = ['initiatives','loans','transactions','customers','market','hft','options','liquidity']
rows = []
data = {}
for n in names:
    d = load(n); data[n] = d
    rows.append({'dataset': n, 'rows': len(d), 'columns': d.shape[1],
                 'missing cells': int(d.isna().sum().sum()),
                 'duplicate rows': int(d.duplicated().sum()),
                 'numeric cols': int(d.select_dtypes('number').shape[1])})
inventory = pd.DataFrame(rows).set_index('dataset')
inventory

,rows,columns,missing cells,duplicate rows,numeric cols
dataset,,,,,
initiatives,180,14,0,0,12
loans,5000,14,0,0,13
transactions,30000,12,0,0,8
customers,5000,13,0,0,13
market,21855,15,0,0,12
hft,120000,33,0,0,27
options,10000,11,0,0,8
liquidity,1216,13,0,0,12


> ### 📘 Explanation — cell 7
> **Reading.** No missing/duplicates → no imputation. Next step: check target class balance.

No missing values and no duplicate rows anywhere, so cleaning is about ranges and types, not imputation. Next: target variables and class balance.

> ### 📘 Explanation — cell 8
> **What it does.** For the five supervised targets (initiative success, loan default, fraud, churn, HFT price move)
> prints the share of each class with `value_counts(normalize=True)`.
>
> **Why this method.** Class balance decides the **metric**, **split** and **threshold** strategy for every classifier.
> It must be known before choosing models.
>
> **Input → Output.** Input: the target column of each dataset. Output: initiatives 77% success; loans 19.9% default;
> fraud **2.3%**; churn **1.8%**; HFT DOWN 41.9% / UP 41.7% / FLAT 16.4%.
>
> **Why the output matters compared with alternatives.**
> - With 2.3% fraud a model that always says "not fraud" gets **97.7% accuracy** but catches nothing → accuracy is
>   useless; use **PR-AUC, recall, cost-based threshold** (notebook 03), class weights / SMOTE.
> - With 20% defaults, **ROC-AUC/Gini/KS** (industry credit metrics) are appropriate (notebook 02).
> - HFT is roughly balanced 3-class → **macro-F1 / balanced accuracy** and per-class recall (notebook 07).

In [4]:
targets = {'initiatives': 'Historical_Success', 'loans': 'Default_Flag', 'transactions': 'Fraud_Flag',
           'customers': 'Churn_Flag', 'hft': 'Price_Move_Class'}
out = []
for n, t in targets.items():
    vc = data[n][t].value_counts(normalize=True).round(3)
    out.append({'dataset': n, 'target': t, 'class balance': dict(vc)})
pd.DataFrame(out).set_index('dataset')

,target,class balance
dataset,,
initiatives,Historical_Success,"{1: 0.767, 0: 0.233}"
loans,Default_Flag,"{0: 0.801, 1: 0.199}"
transactions,Fraud_Flag,"{0: 0.977, 1: 0.023}"
customers,Churn_Flag,"{0: 0.982, 1: 0.018}"
hft,Price_Move_Class,"{'DOWN': 0.419, 'UP': 0.417, 'FLAT': 0.164}"


> ### 📘 Explanation — cell 9
> **Interpretation cell** – summarises the implication above (rare classes ⇒ PR-AUC and cost thresholds, not accuracy).

**Reading this.** Fraud (2.3%) and churn (1.8%) are rare, which is why accuracy is meaningless there and PR-AUC, recall and cost-based thresholds are needed. Defaults (20%) are moderately imbalanced. HFT classes are fairly balanced, with FLAT (16%) the smallest. Initiatives succeed 77% of the time.

> ### 📘 Explanation — cell 10
> **Section header** – are money columns in rupees or lakh? A wrong unit would make every ₹ number in the report wrong by 10⁵.

## 3. Units: are the money columns rupees or lakh?

> ### 📘 Explanation — cell 11
> **What it does.** Prints the mean annual income and mean loan amount in the loans file, the total loan book in crore
> (using `cr()` = divide by 10⁷), and min/mean/max of the initiative money columns.
>
> **Why this method.** A *sanity check by magnitude*: compute an average and ask "is this realistic?". It is cheaper and
> more reliable than trusting the README blindly.
>
> **Input → Output.** Output: average income ≈ 1,051,769, average loan ≈ 1,279,306, total book ≈ **₹640 crore**;
> initiative investments 31.5 – 296.5.
>
> **How to read it.** If income were in lakh, the average borrower would earn ~₹1 lakh crore/year — absurd. So loans
> and customers are in **rupees**. Initiatives (31–297) fit **₹ crore** relative to a ₹1,000 cr budget. The next cell
> records the decision.

In [5]:
loans, init = data['loans'], data['initiatives']
print('Average annual income in loans file:', f"{loans.Annual_Income.mean():,.0f}")
print('Average loan amount:               ', f"{loans.Loan_Amount.mean():,.0f}")
print('Total loan book (rupees crore):    ', f"{cr(loans.Loan_Amount.sum()):,.0f}")
print()
print(init[['Initial_Investment','Revenue_Year1','Operating_Cost']].describe().round(1).loc[['mean','min','max']])

Average annual income in loans file: 1,051,769
Average loan amount:                1,279,306
Total loan book (rupees crore):     640

      Initial_Investment  Revenue_Year1  Operating_Cost
mean               157.5           56.7            35.9
min                 31.5            9.5             5.2
max                296.5          137.8            95.7


> ### 📘 Explanation — cell 12
> **Decision recorded.** Loans/customers = rupees; initiatives = ₹ crore. Important argument for the report: the
> Fund/Review/Reject decision in 01 uses **ratios** (NPV/investment, IRR, profitability index), so it is unit-independent
> — even if the unit assumption were wrong, the ranking would not change.

**Decision.** The README says money is in INR lakh unless the column context says otherwise. Read as lakh, the average borrower would earn about ₹1,000 billion a year, which is absurd, so the loan and customer files are in **rupees** (average income about ₹10.5 lakh). The strategic initiatives file has much smaller numbers (investments between about 31 and 297) and is plausibly in **₹ crore**, which fits a ₹1,000 crore budget. Notebook 01 treats it as crore; the Fund/Review/Reject decision depends only on ratios, so it does not change if the unit is different.

The loan book in the file is about ₹640 crore, which is a useful scale check against the ₹1,000 crore to allocate.

> ### 📘 Explanation — cell 13
> **Section header** – which files are time series? This decides random vs time-based validation.

## 4. Time structure: which files need time-based splits?

> ### 📘 Explanation — cell 14
> **What it does.** Prints the date range of market data (and number of tickers), transactions, liquidity, the HFT
> structure (trading days, stocks, how many unique timestamps) and the options expiry range.
>
> **Why this method.** Before choosing cross-validation, you must know if rows are ordered in time. Random K-fold on a
> time series leaks the future into training ("look-ahead bias") and gives over-optimistic scores.
>
> **Input → Output.** Output: market 2021-01 → 2026-08, 15 tickers; transactions 2025-01 → 2026-08; HFT 20 days ×
> 10 stocks but only **7,506 unique timestamps for 120,000 rows** (timestamps are to the minute); liquidity
> 2022-01 → 2026-08; options expiries 2026-09 → 2028-08.
>
> **Why it matters.** The HFT finding is subtle: many rows share the same minute, so you cannot order them by time
> within the minute → split by **trading day** instead (notebook 07).

In [6]:
mk, tx, hft, liq, opt = data['market'], data['transactions'], data['hft'], data['liquidity'], data['options']
print('market   : ', mk.Date.min(), 'to', mk.Date.max(), '|', mk.Ticker.nunique(), 'tickers')
print('transactions:', tx.Timestamp.min(), 'to', tx.Timestamp.max())
print('hft      : ', hft.Trading_Day.nunique(), 'trading days,', hft.Stock.nunique(), 'stocks; timestamp unique values:', hft.Timestamp.nunique(), 'of', len(hft), 'rows')
print('liquidity:', liq.Date.min(), 'to', liq.Date.max())
print('options  : expiry range', opt.Expiry.min(), 'to', opt.Expiry.max())

market   :  2021-01-29 to 2026-08-31 | 15 tickers
transactions: 2025-01-01 00:00:00.000000000 to 2026-08-31 00:00:00.000000000
hft      :  20 trading days, 10 stocks; timestamp unique values: 7506 of 120000 rows
liquidity: 2022-01-03 to 2026-08-31
options  : expiry range 2026-09-13 to 2028-08-31


> ### 📘 Explanation — cell 15
> **Validation plan (key viva point).**
> | Data | Split | Reason |
> |---|---|---|
> | Market, liquidity, transactions | time-based / walk-forward | avoid look-ahead bias |
> | HFT | by trading day | timestamps not unique within a minute |
> | Loans, customers, initiatives | stratified random K-fold | cross-sectional, keeps class ratio in each fold |
> | Options | random (plus tougher group split discussed) | cross-section of contracts |

**Consequences for validation**
- Market, liquidity and transactions: split by **time**, never randomly.
- HFT: timestamps only go to the minute, so rows within a minute cannot be ordered by time. Split by **trading day** and keep the file order.
- Loans, customers, initiatives: stratified random splits are fine (cross-sectional data).
- Options: random split is acceptable for a cross-section, but a split by underlying or expiry is a tougher and fairer test; notebook 08 discusses this.

> ### 📘 Explanation — cell 16
> **Section header** – leakage list. Leakage = a feature that would not be available at prediction time, or that encodes the target.

## 5. Leakage watch-list (decided now, applied in each notebook)

> ### 📘 Explanation — cell 17
> **What it does.** Builds a small table of columns that must be excluded or handled carefully in each dataset.
>
> **Why this method.** Writing the leakage list *before* modelling is good practice: it prevents a model from looking
> excellent for the wrong reason.
>
> **Output (and the reason for each).**
> - **HFT** `Future_Return_100ms`, `Future_Price_100ms` → these *are* the answer (the class is derived from them).
> - **Customers** `Estimated_CLV` → built from retention, i.e. already contains churn information.
> - **Options** `Black_Scholes_Price` → it is the benchmark; feeding it to the ML model would make the comparison
>   meaningless.
> - **Market** same-day `Return` → the target must be **next-period** return (shift by −1).
> - **Loans** `Interest_Rate` → legitimately known at origination, but it already reflects the lender's risk view;
>   notebook 02 tests with and without it.
>
> **Why it matters.** "How did you avoid leakage?" is a very common report/viva question — this table is the answer.

In [7]:
pd.DataFrame([
 ['hft', 'Future_Return_100ms, Future_Price_100ms', 'They are the answer to Price_Move_Class'],
 ['customers', 'Estimated_CLV', 'Derived from customer value and retention, so it is related to churn. Keep out of churn features'],
 ['options', 'Black_Scholes_Price', 'Do not feed it to the model when comparing ML against Black-Scholes'],
 ['market', 'Return (same-day)', 'Target must be the next-period return, built by shifting'],
 ['loans', 'Interest_Rate', 'Known at origination so allowed, but it already embeds the lender\'s own risk view. Notebook 02 tests the model with and without it'],
], columns=['dataset','column(s)','issue']).set_index('dataset')

,column(s),issue
dataset,,
hft,"Future_Return_100ms, Future_Price_100ms",They are the answer to Price_Move_Class
customers,Estimated_CLV,"Derived from customer value and retention, so ..."
options,Black_Scholes_Price,Do not feed it to the model when comparing ML ...
market,Return (same-day),"Target must be the next-period return, built b..."
loans,Interest_Rate,"Known at origination so allowed, but it alread..."


> ### 📘 Explanation — cell 18
> **Brief vs data mismatches.** No `Loan_ID` (row index used), HFT class is FLAT not "No Move", liquidity file not in
> README but covered in notebook 09, module numbering differs from notebook numbering. Stating these shows careful
> reading of the data. **Status:** setup done when this runs top-to-bottom.

## 6. What the brief and the files disagree on
- Loans have no `Loan_ID` column (only `Customer_ID`), so the notebooks use the row index as the loan ID.
- The HFT class is called **FLAT**, not No Move, and the target is `Future_Return_100ms`.
- `nova_liquidity.csv` is not listed in the README, and the brief has no module for it, but liquidity forecasting and stress testing are listed as skills. Notebook 09 covers it.
- Module numbers jump (1–6, 9, 10, 11). The notebooks are numbered 01–10 in the order of work, and module numbers are given in each title.

## Status
Setup is complete when this notebook runs top to bottom without errors. Next: **01 Strategic initiatives** or **02 Credit risk**.